# Experiment 13: Visual Annotation Comparison — Human A vs Human B vs LLM

This notebook provides **visual** side-by-side comparison of the raw KC gap annotations from:
- **Human A** (Pranay Ghuge)
- **Human B** (Arundhati Das)
- **LLM** (Exp11 Enriched V2 — best performing model)

Visualizations:
1. Per-KC tagging frequency across all three annotators
2. Per-student annotation heatmaps (problems × KCs)
3. Agreement pattern breakdown per problem
4. Annotator disagreement explorer

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Set, Tuple

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

ROOT = Path.cwd()
if not (ROOT / "lib").exists() and (ROOT.parent / "lib").exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

print(f"Working directory: {os.getcwd()}")

In [ ]:
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

def load_annotations(filepath: str) -> Tuple[str, Dict[str, Set[str]]]:
    path = Path(filepath)
    if not path.exists():
        print(f"Warning: File not found {filepath}")
        return path.stem, {}
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    student_id = data.get("student_id", data.get("studentId", "unknown"))
    parsed: Dict[str, Set[str]] = {}
    for pid, val in data.get("annotations", {}).items():
        comp_pid = f"{student_id}_{pid}"
        if isinstance(val, dict) and "gaps" in val:
            gaps = val.get("gaps")
            parsed[comp_pid] = set(gaps) if isinstance(gaps, list) else set()
        elif isinstance(val, list):
            parsed[comp_pid] = set(val)
        else:
            parsed[comp_pid] = set()
        parsed[comp_pid] = {t for t in parsed[comp_pid] if t in EXACT_KC_TAGS}
    return path.stem, parsed

def merge_rater_files(filepaths: List[str], label: str) -> Tuple[str, Dict[str, Set[str]]]:
    merged = {}
    for fp in filepaths:
        _, anns = load_annotations(fp)
        for comp_pid, kcs in anns.items():
            if comp_pid in merged:
                merged[comp_pid].update(kcs)
            else:
                merged[comp_pid] = set(kcs)
    return label, merged

In [ ]:
HUMAN_A_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json",
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json",
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json",
]
HUMAN_B_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json",
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json",
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json",
]
LLM_FILES = [
    "results/human_validation/llm_enriched_annotations_v2_10155.json",
    "results/human_validation/llm_enriched_annotations_v2_14475.json",
    "results/human_validation/llm_enriched_annotations_v2_14476.json",
]

_, anns_ha = merge_rater_files(HUMAN_A_FILES, "Human A")
_, anns_hb = merge_rater_files(HUMAN_B_FILES, "Human B")
_, anns_llm = merge_rater_files(LLM_FILES, "LLM (Exp11 Enriched V2)")

# Common problems annotated by all three
common_pids = sorted(
    set(anns_ha.keys()) & set(anns_hb.keys()) & set(anns_llm.keys())
)
# Remove trivially empty problems (all three tagged zero gaps)
common_pids = [
    pid for pid in common_pids
    if any(len(a.get(pid, set())) > 0 for a in [anns_ha, anns_hb, anns_llm])
]
print(f"Common non-empty problems: {len(common_pids)}")

## 1. Per-KC Tagging Frequency
How often each KC was flagged as a gap by each annotator across all problems.

In [ ]:
kc_counts = {}
for kc in EXACT_KC_TAGS:
    kc_counts[kc] = {
        "Human A": sum(1 for pid in common_pids if kc in anns_ha.get(pid, set())),
        "Human B": sum(1 for pid in common_pids if kc in anns_hb.get(pid, set())),
        "LLM": sum(1 for pid in common_pids if kc in anns_llm.get(pid, set())),
    }

df_kc = pd.DataFrame(kc_counts).T.reset_index().rename(columns={"index": "KC"})
df_kc_melted = df_kc.melt(id_vars="KC", var_name="Annotator", value_name="Count")

COLORS = {"Human A": "#4C72B0", "Human B": "#DD8452", "LLM": "#55A868"}

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(EXACT_KC_TAGS))
width = 0.27

for i, (annotator, color) in enumerate(COLORS.items()):
    vals = df_kc[annotator].values
    bars = ax.bar(x + (i - 1) * width, vals, width, label=annotator, color=color, alpha=0.88)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(v), ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(EXACT_KC_TAGS, rotation=40, ha='right', fontsize=9)
ax.set_ylabel("Number of Problems Tagged", fontsize=11)
ax.set_title("KC Tagging Frequency: Human A vs Human B vs LLM", fontsize=14, pad=12)
ax.legend(fontsize=10)
ax.set_xlim(-0.6, len(EXACT_KC_TAGS) - 0.4)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("results/visualizations/exp13_kc_frequency.png", dpi=150)
plt.show()

## 2. Per-Student Annotation Heatmaps
For each student, a heatmap shows which KCs were flagged per problem (rows = problems, cols = KCs).  
Three side-by-side subplots: Human A | Human B | LLM.

In [ ]:
# Group problems by student
students = sorted({pid.split("_")[0] for pid in common_pids})
print("Students:", students)

annotators = {
    "Human A": anns_ha,
    "Human B": anns_hb,
    "LLM (Exp11 Enriched V2)": anns_llm,
}

for student in students:
    s_pids = sorted([pid for pid in common_pids if pid.startswith(f"{student}_")])
    prob_labels = [pid.split("_", 1)[1] for pid in s_pids]

    fig, axes = plt.subplots(1, 3, figsize=(22, max(5, len(s_pids) * 0.38 + 2)),
                             sharey=True, constrained_layout=True)
    fig.suptitle(f"Student {student} — Annotation Heatmaps", fontsize=14, y=1.02)

    for ax, (ann_name, ann_dict) in zip(axes, annotators.items()):
        matrix = np.zeros((len(s_pids), len(EXACT_KC_TAGS)), dtype=int)
        for r, pid in enumerate(s_pids):
            for c, kc in enumerate(EXACT_KC_TAGS):
                matrix[r, c] = 1 if kc in ann_dict.get(pid, set()) else 0

        cmap = ListedColormap(["#f0f4ff", "#2c6fad"])
        sns.heatmap(
            matrix, ax=ax,
            xticklabels=EXACT_KC_TAGS,
            yticklabels=prob_labels,
            cmap=cmap, vmin=0, vmax=1,
            linewidths=0.4, linecolor="#cccccc",
            cbar=False
        )
        ax.set_title(ann_name, fontsize=11, pad=8)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=7.5)

    legend_els = [
        mpatches.Patch(facecolor="#2c6fad", label="Gap Tagged"),
        mpatches.Patch(facecolor="#f0f4ff", edgecolor="#aaa", label="No Gap"),
    ]
    fig.legend(handles=legend_els, loc="lower center", ncol=2,
               bbox_to_anchor=(0.5, -0.04), fontsize=10)

    plt.savefig(f"results/visualizations/exp13_heatmap_student_{student}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

## 3. Agreement Pattern Breakdown
For every (problem × KC) cell, classify agreement into 5 categories:
- **All Agree — Gap**: all three flagged it
- **All Agree — No Gap**: all three said no gap
- **Humans Only**: both humans tagged it, LLM did not
- **LLM Only**: LLM tagged it, both humans did not
- **Partial Disagreement**: mixed (1 of 2 humans + LLM or other combos)

In [ ]:
agreement_counts = {"All Agree — Gap": 0, "Humans Only": 0, "LLM Only": 0,
                    "Partial Disagreement": 0, "All Agree — No Gap": 0}

rows = []
for pid in common_pids:
    for kc in EXACT_KC_TAGS:
        ha = kc in anns_ha.get(pid, set())
        hb = kc in anns_hb.get(pid, set())
        llm = kc in anns_llm.get(pid, set())
        n_yes = sum([ha, hb, llm])

        if n_yes == 3:
            cat = "All Agree — Gap"
        elif n_yes == 0:
            cat = "All Agree — No Gap"
        elif ha and hb and not llm:
            cat = "Humans Only"
        elif llm and not ha and not hb:
            cat = "LLM Only"
        else:
            cat = "Partial Disagreement"

        agreement_counts[cat] += 1
        rows.append({"pid": pid, "kc": kc, "ha": ha, "hb": hb, "llm": llm, "category": cat})

df_agree = pd.DataFrame(rows)

CAT_COLORS = {
    "All Agree — Gap":     "#2ca02c",
    "All Agree — No Gap":  "#d3d3d3",
    "Humans Only":         "#1f77b4",
    "LLM Only":            "#ff7f0e",
    "Partial Disagreement":"#9467bd",
}

cats = [c for c in CAT_COLORS if c != "All Agree — No Gap"]
vals = [agreement_counts[c] for c in cats]
colors = [CAT_COLORS[c] for c in cats]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart (excluding "All Agree — No Gap" for visual clarity)
wedges, texts, autotexts = axes[0].pie(
    vals, labels=cats, colors=colors, autopct='%1.1f%%',
    startangle=140, pctdistance=0.78,
    textprops={'fontsize': 9}
)
axes[0].set_title("Agreement Breakdown\n(excluding trivial No-Gap agreement)", fontsize=12)

# Bar chart — all categories
all_cats = list(CAT_COLORS.keys())
all_vals = [agreement_counts[c] for c in all_cats]
all_colors = [CAT_COLORS[c] for c in all_cats]
bars = axes[1].barh(all_cats, all_vals, color=all_colors, alpha=0.88)
for bar, v in zip(bars, all_vals):
    axes[1].text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
                 f"{v:,}", va='center', fontsize=10)
axes[1].set_xlabel("Number of (Problem × KC) Cells", fontsize=10)
axes[1].set_title("Agreement Counts (All Categories)", fontsize=12)
axes[1].set_xlim(0, max(all_vals) * 1.15)
axes[1].xaxis.grid(True, linestyle='--', alpha=0.5)
axes[1].set_axisbelow(True)

plt.suptitle("Human A vs Human B vs LLM — Agreement Patterns", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("results/visualizations/exp13_agreement_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

total = sum(agreement_counts.values())
print(f"Total (problem × KC) cells: {total}")
for cat, cnt in agreement_counts.items():
    print(f"  {cat:<30}: {cnt:>5}  ({cnt/total*100:.1f}%)")

## 4. Agreement Heatmap — Per KC
For each KC, what fraction of problems fall into each agreement category?

In [ ]:
kc_cat_matrix = pd.crosstab(df_agree['kc'], df_agree['category'])
# Reorder columns and rows
ordered_cats = ["All Agree — Gap", "Humans Only", "LLM Only", "Partial Disagreement", "All Agree — No Gap"]
kc_cat_matrix = kc_cat_matrix.reindex(columns=[c for c in ordered_cats if c in kc_cat_matrix.columns])
kc_cat_matrix = kc_cat_matrix.reindex(EXACT_KC_TAGS)

# Normalize to proportions
kc_cat_pct = kc_cat_matrix.div(kc_cat_matrix.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 7))
kc_cat_pct_plot = kc_cat_pct.drop(columns=["All Agree — No Gap"], errors='ignore')
kc_cat_pct_plot.plot(kind='barh', stacked=True, ax=ax,
                     color=[CAT_COLORS[c] for c in kc_cat_pct_plot.columns],
                     width=0.72, alpha=0.9)

ax.set_xlabel("Percentage of Problems (%)", fontsize=11)
ax.set_title("Per-KC Agreement Breakdown (excl. All-No-Gap)", fontsize=13, pad=10)
ax.legend(loc='lower right', fontsize=9, framealpha=0.8)
ax.set_xlim(0, 100)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("results/visualizations/exp13_kc_agreement_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Disagreement Explorer
Show all (problem, KC) cells where the three annotators **disagree** — ranked by how contentious they are.

In [ ]:
df_disagree = df_agree[df_agree['category'] != "All Agree — No Gap"].copy()
df_disagree['student'] = df_disagree['pid'].apply(lambda p: p.split('_')[0])
df_disagree['problem'] = df_disagree['pid'].apply(lambda p: p.split('_', 1)[1])

# Show disagreement cases (not all-agree-gap or all-agree-no-gap)
df_conflict = df_disagree[~df_disagree['category'].isin(["All Agree — Gap", "All Agree — No Gap"])].copy()
df_conflict['Human A'] = df_conflict['ha'].map({True: '✓ Gap', False: '— No Gap'})
df_conflict['Human B'] = df_conflict['hb'].map({True: '✓ Gap', False: '— No Gap'})
df_conflict['LLM'] = df_conflict['llm'].map({True: '✓ Gap', False: '— No Gap'})

display_cols = ['student', 'problem', 'kc', 'Human A', 'Human B', 'LLM', 'category']
df_display = df_conflict[display_cols].sort_values(['student', 'problem', 'kc']).reset_index(drop=True)

print(f"Total disagreement cells: {len(df_display)}")
print(f"Breakdown:")
print(df_display['category'].value_counts().to_string())
print()

# Style and display
def color_category(val):
    palette = {
        "Humans Only":          "background-color: #cce5ff",
        "LLM Only":             "background-color: #ffe0b2",
        "Partial Disagreement": "background-color: #ede7f6",
    }
    return palette.get(val, "")

df_display.style.applymap(color_category, subset=['category'])

In [ ]:
# Summary table: per-student disagreement rates
summary_rows = []
for student in students:
    s_pids = [pid for pid in common_pids if pid.startswith(f"{student}_")]
    s_df = df_agree[df_agree['pid'].isin(s_pids)]
    total_cells = len(s_df)
    row = {"Student": student, "Total Cells": total_cells}
    for cat in ordered_cats:
        cnt = (s_df['category'] == cat).sum()
        row[cat] = f"{cnt} ({cnt/total_cells*100:.0f}%)"
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Student')
print("\n=== Per-Student Agreement Summary ===")
df_summary